# 05 · Afinar o TF-IDF: regularização e etiquetas frequentes

Spike E4-S4 do [`docs/backlog.md`](../docs/backlog.md). Os números que saem daqui vão para o [`docs/qualidade_dados.md`](../docs/qualidade_dados.md).

**Porquê:** no E4-H1 (notebook 04) o TF-IDF bateu a régua, mas a 1.ª sugestão era quase sempre uma das 5 etiquetas mais frequentes e o F1 macro ficou quase zero. Duas suspeitas, cada uma com uma correção:

- **Regularização forte demais.** A *regularização* é um travão que impede o modelo de dar pesos grandes às palavras, para não decorar o treino. No `SGDClassifier` o travão chama-se `alpha`, e no E4-H1 ficou no valor por defeito, 0,0001. Uma etiqueta com 10 exemplos aparece em cerca de 0,00002 dos 478 mil textos de treino, menos do que o travão. A hipótese (não medida aqui) é que o modelo dessa etiqueta "poupa" mais se não aprender palavras nenhumas. Ficam só os interceptos (a pontuação de base de cada etiqueta, igual para todos os textos), e ganham as etiquetas frequentes. A documentação do scikit-learn recomenda procurar o `alpha` entre 0,1 e 0,000001.
- **Vantagem das etiquetas frequentes.** Menon et al. (2021, *logit adjustment*) ordenam por **pontuação − τ · log(frequência da etiqueta no treino)**. Exemplo: com τ = 1, uma etiqueta com frequência 0,1 ganha +2,3 e uma com 0,5 ganha só +0,7. Com τ = 0 nada muda. Não obriga a treinar de novo.

**O que se treinou** (no pod, com o treino completo; o melhor `alpha` depende do número de exemplos, por isso uma amostra no portátil não servia): `alpha` = 0,0001, 0,00001 e 0,000001 (e depois 0,0000001, porque o 0,000001 ganhou e era o extremo da lista), e para cada um as sugestões com τ = 0, 0,25, 0,5 e 1. Os ficheiros estão em `metrics/e4_s4/alpha_<alpha>/`.

**Regra de escolha (escrita no backlog antes de correr):** primeiro o `alpha` com τ = 0, depois o τ com esse `alpha`. Em cada passo começa-se pela opção por defeito (`alpha` = 0,0001; τ = 0) e compara-se cada candidato com a opção em vigor, pela ordem da lista, com a regra do E4-T1: decide o top-5 no código completo, o top-5 na classe é a guarda, e uma diferença só conta se o bootstrap emparelhado a mostrar em pelo menos 95% das amostras. Se o candidato ganhar, passa a ser a opção em vigor.

O teste (novembro e dezembro de 2025) não se lê aqui.

In [1]:
import json
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / "src"))
import baselines as bl
import comparison as cmp
import evaluation as ev

DATA = Path.cwd().parent / "data" / "processed"
METRICS = Path.cwd().parent / "metrics"
ALPHAS = ["1e-4", "1e-5", "1e-6", "1e-7"]  # a 1.ª é a opção por defeito
TAUS = [0.0, 0.25, 0.5, 1.0]       # a 1.ª é a opção por defeito

info = {a: json.loads((METRICS / "e4_s4" / f"alpha_{a}" / "treino.json").read_text(encoding="utf-8")) for a in ALPHAS}
N_MIN = info[ALPHAS[0]]["n_min"]
if any(i["n_min"] != N_MIN for i in info.values()):
    raise ValueError("os treinos não usaram todos o mesmo N")

treino = pd.read_parquet(DATA / "treino.parquet", columns=["codigo"])
val = pd.read_parquet(DATA / "validacao.parquet", columns=["idcontrato", "codigos", "visto_no_treino"])
print(f"treino: {len(treino):,} exemplos · validação: {len(val):,} contratos · N = {N_MIN}")

treino: 478,384 exemplos · validação: 23,945 contratos · N = 10


## Os treinos

Uma linha por `alpha`, lida do `treino.json` de cada treino. **Épocas** são as passagens completas pelo treino; um `alpha` mais pequeno costuma precisar de mais.

In [2]:
pd.DataFrame({a: {"alpha": f'{i["alpha"]:g}', "épocas": i["n_iter"][0], "convergiu": i["convergiu"],
                   "etiquetas": i["etiquetas"], "vocabulário": i["vocabulario"],
                   "treinar (s)": i["tempo_vetorizar_treinar_s"], "sugerir (s)": i["tempo_sugerir_s"],
                   "pico de memória (MiB)": i["pico_rss_mb"]} for a, i in info.items()}).T

,alpha,épocas,convergiu,etiquetas,vocabulário,treinar (s),sugerir (s),pico de memória (MiB)
1e-4,0.0001,6,True,3000,402084,380.34,32.47,9947
1e-5,1e-05,7,True,3000,402084,423.84,33.4,9989
1e-6,1e-06,8,True,3000,402084,386.98,33.66,10000
1e-7,1e-07,12,True,3000,402084,463.34,75.5,9990


## As sugestões

Um ficheiro por (`alpha`, τ). As verificações são as do notebook 04: uma linha por contrato da validação, alinhada pelo `idcontrato` (nunca pela posição), com 5 códigos distintos de 8 dígitos. Usam `raise` e não `assert`, porque o Python ignora os `assert` quando corre com `-O`.

In [3]:
CODIGO = re.compile(r"\d{8}")


def ficheiro(alpha, tau):
    nome = "sugestoes_validacao.parquet" if tau == 0 else f"sugestoes_validacao_tau{tau:g}.parquet"
    return METRICS / "e4_s4" / f"alpha_{alpha}" / nome


def ler(path):
    sug = pd.read_parquet(path, columns=["idcontrato", "sugestoes"])
    sug["idcontrato"] = sug["idcontrato"].astype(str)
    if sug["idcontrato"].duplicated().any() or set(sug["idcontrato"]) != set(val["idcontrato"]):
        raise ValueError(f"{path.name}: os contratos não são os da validação, um por linha")
    s = sug.set_index("idcontrato")["sugestoes"].reindex(val["idcontrato"]).map(list)
    s.index = val.index
    if not s.map(lambda l: len(set(l)) == ev.TOP_K and all(CODIGO.fullmatch(c) for c in l)).all():
        raise ValueError(f"{path.name}: há contratos sem {ev.TOP_K} códigos distintos de 8 dígitos")
    return s


opcoes = {(a, t): ler(ficheiro(a, t)) for a in ALPHAS for t in TAUS}
print(f"{len(opcoes)} opções lidas · ok")

16 opções lidas · ok


**Repetível?** O treino com `alpha` = 0,0001 e τ = 0 é o mesmo modelo do E4-H1 (mesmos dados, mesma semente). Se as sugestões não forem iguais às do E4-H1, o treino não é repetível e nenhuma comparação aqui vale.

In [4]:
e4h1 = ler(METRICS / "e4_h1" / "sugestoes_validacao.parquet")
iguais = (e4h1 == opcoes[("1e-4", 0.0)]).mean()
print(f"sugestões iguais às do E4-H1: {iguais:.2%}")
if iguais < 1:
    raise ValueError("o treino com alpha = 0,0001 e τ = 0 não repetiu o E4-H1")

sugestões iguais às do E4-H1: 100.00%


## Todas as opções lado a lado

A régua é a do E4-T2, com o mesmo N. Além das métricas da regra (top-5 no código completo e na classe), mostram-se duas que não entram na decisão:
- **F1 macro** no código completo: a média do F1 de cada etiqueta. Sobe quando o modelo acerta também etiquetas pouco frequentes.
- **1.ª sugestão na régua:** a parte dos contratos em que a 1.ª sugestão é uma das 5 etiquetas da régua. No E4-H1 era muito alta: o sintoma que este spike tenta corrigir.

In [5]:
top = bl.most_common_labels(treino["codigo"], n_min=N_MIN)
regua = bl.suggest_most_common(top, val.index)
novo = ~val["visto_no_treino"]


def linha(s):
    rep = ev.report(s, val["codigos"], val["visto_no_treino"])
    return {
        "top-5 código": rep.loc[("todos", "codigo"), "top5"],
        "top-5 classe": rep.loc[("todos", "classe"), "top5"],
        "top-5 código (novo)": rep.loc[("texto novo", "codigo"), "top5"],
        "top-5 classe (novo)": rep.loc[("texto novo", "classe"), "top5"],
        "top-1 código": rep.loc[("todos", "codigo"), "top1"],
        "F1 macro código": rep.loc[("todos", "codigo"), "f1_macro"],
        "1.ª sugestão na régua": s.map(lambda l: l[0] in top).mean(),
    }


tabela = pd.DataFrame({**{(f"alpha {a}", f"τ {t:g}"): linha(s) for (a, t), s in opcoes.items()},
                       ("régua", "-"): linha(regua)}).T
tabela.style.format({c: "{:.1%}" for c in tabela.columns if c != "F1 macro código"} | {"F1 macro código": "{:.4f}"})

## Passo 1: escolher o `alpha` (com τ = 0)

Cada candidato contra a opção em vigor, com o bootstrap emparelhado (1 000 amostras da validação, a mesma amostra para os dois). `cmp.choose` aplica a regra do E4-T1: devolve `"b"` se o candidato ganhar e `"a"` se ficar a opção em vigor.

In [6]:
def duelo(em_vigor, candidato):
    """Top-5 no código e na classe, em vigor (a) contra candidato (b), e a decisão."""
    res = {}
    for medida in ("codigo", "classe"):
        a = cmp.weighted_mean(ev.hits(em_vigor, val["codigos"], medida, ev.TOP_K))
        b = cmp.weighted_mean(ev.hits(candidato, val["codigos"], medida, ev.TOP_K))
        res[medida] = cmp.paired_bootstrap(a, b, len(val))
    return res, cmp.choose(res["codigo"], res["classe"], default="a")


def mostrar(nome_a, nome_b, res, escolha):
    for medida, r in res.items():
        print(f"  {medida:7s} {nome_a}: {r['a']:.2%}  {nome_b}: {r['b']:.2%}  "
              f"diferença {r['b'] - r['a']:+.2%}  candidato melhor em {r['b_melhor']:.0%} das amostras")
    print(f"  -> fica {nome_b if escolha == 'b' else nome_a}")


alpha = ALPHAS[0]
for cand in ALPHAS[1:]:
    res, escolha = duelo(opcoes[(alpha, 0.0)], opcoes[(cand, 0.0)])
    print(f"alpha {alpha} (em vigor) contra alpha {cand}:")
    mostrar(f"alpha {alpha}", f"alpha {cand}", res, escolha)
    if escolha == "b":
        alpha = cand
print(f"\nalpha escolhido: {alpha}")

alpha 1e-4 (em vigor) contra alpha 1e-5:
  codigo  alpha 1e-4: 43.19%  alpha 1e-5: 61.55%  diferença +18.37%  candidato melhor em 100% das amostras
  classe  alpha 1e-4: 37.91%  alpha 1e-5: 60.82%  diferença +22.92%  candidato melhor em 100% das amostras
  -> fica alpha 1e-5


alpha 1e-5 (em vigor) contra alpha 1e-6:
  codigo  alpha 1e-5: 61.55%  alpha 1e-6: 75.30%  diferença +13.74%  candidato melhor em 100% das amostras
  classe  alpha 1e-5: 60.82%  alpha 1e-6: 78.42%  diferença +17.60%  candidato melhor em 100% das amostras
  -> fica alpha 1e-6


alpha 1e-6 (em vigor) contra alpha 1e-7:
  codigo  alpha 1e-6: 75.30%  alpha 1e-7: 76.91%  diferença +1.61%  candidato melhor em 100% das amostras
  classe  alpha 1e-6: 78.42%  alpha 1e-7: 82.20%  diferença +3.77%  candidato melhor em 100% das amostras
  -> fica alpha 1e-7

alpha escolhido: 1e-7


## Passo 2: escolher o τ (com o `alpha` escolhido)

In [7]:
tau = TAUS[0]
for cand in TAUS[1:]:
    res, escolha = duelo(opcoes[(alpha, tau)], opcoes[(alpha, cand)])
    print(f"τ {tau:g} (em vigor) contra τ {cand:g}:")
    mostrar(f"τ {tau:g}", f"τ {cand:g}", res, escolha)
    if escolha == "b":
        tau = cand
print(f"\nescolhido: alpha {alpha}, τ {tau:g}")

τ 0 (em vigor) contra τ 0.25:
  codigo  τ 0: 76.91%  τ 0.25: 76.94%  diferença +0.04%  candidato melhor em 62% das amostras
  classe  τ 0: 82.20%  τ 0.25: 82.94%  diferença +0.75%  candidato melhor em 100% das amostras
  -> fica τ 0.25


τ 0.25 (em vigor) contra τ 0.5:
  codigo  τ 0.25: 76.94%  τ 0.5: 76.42%  diferença -0.52%  candidato melhor em 0% das amostras
  classe  τ 0.25: 82.94%  τ 0.5: 82.88%  diferença -0.06%  candidato melhor em 30% das amostras
  -> fica τ 0.25


τ 0.25 (em vigor) contra τ 1:
  codigo  τ 0.25: 76.94%  τ 1: 71.72%  diferença -5.22%  candidato melhor em 0% das amostras
  classe  τ 0.25: 82.94%  τ 1: 80.96%  diferença -1.98%  candidato melhor em 0% das amostras
  -> fica τ 0.25

escolhido: alpha 1e-7, τ 0.25


**Uma verificação à regra em dois passos.** A regra escolhe primeiro o `alpha` (com τ = 0) e só depois o τ, por isso não compara todas as combinações entre si. Na tabela acima, `alpha` = 0,000001 com τ = 0,5 tem um top-5 no código completo um pouco maior do que o escolhido. Aqui compara-se essa combinação diretamente com o escolhido, com a mesma regra do E4-T1, para ver se a ordem dos passos mudou o resultado.

In [8]:
rival = ("1e-6", 0.5)
res, escolha = duelo(opcoes[(alpha, tau)], opcoes[rival])
print(f"escolhido (alpha {alpha}, τ {tau:g}) contra alpha {rival[0]}, τ {rival[1]:g}:")
mostrar("escolhido", f"alpha {rival[0]}, τ {rival[1]:g}", res, escolha)

escolhido (alpha 1e-7, τ 0.25) contra alpha 1e-6, τ 0.5:
  codigo  escolhido: 76.94%  alpha 1e-6, τ 0.5: 77.81%  diferença +0.87%  candidato melhor em 100% das amostras
  classe  escolhido: 82.94%  alpha 1e-6, τ 0.5: 81.98%  diferença -0.96%  candidato melhor em 0% das amostras
  -> fica escolhido


## O escolhido contra o E4-H1 e contra a régua

A avaliação completa (as quatro medidas, todos os contratos e texto novo) do escolhido, ao lado do E4-H1. E o critério do E4-H1 repetido para o escolhido: continua a bater a régua no top-5 do código completo e da classe, em todos os contratos e no texto novo?

In [9]:
escolhido = opcoes[(alpha, tau)]
lado = pd.concat({"escolhido": ev.report(escolhido, val["codigos"], val["visto_no_treino"]),
                  "E4-H1": ev.report(e4h1, val["codigos"], val["visto_no_treino"])}, axis=1)
lado.style.format("{:.1%}", subset=[c for c in lado.columns if c[1] in ("top1", "top5")]) \
    .format("{:.4f}", subset=[c for c in lado.columns if c[1] == "f1_macro"]) \
    .format("{:,.0f}", subset=[c for c in lado.columns if c[1] == "contratos"])

In [10]:
linhas = {}
for grupo, mask in (("todos", slice(None)), ("texto novo", novo)):
    comp = cmp.compare(escolhido[mask], regua[mask], val["codigos"][mask])
    for medida in ("codigo", "classe"):
        r = comp.loc[(medida, "top5")]
        linhas[(grupo, medida)] = {"escolhido": r["a"], "régua": r["b"], "conta": r["conta"],
                                   "bate a régua": bool(r["conta"] and r["a"] > r["b"])}
verdict = pd.DataFrame(linhas).T
print(verdict)
print("\nContinua a bater a régua:", "SIM" if verdict["bate a régua"].all() else "NÃO")

                  escolhido     régua conta bate a régua
todos      codigo  0.769416  0.263237  True         True
           classe   0.82943  0.155976  True         True
texto novo codigo  0.735104  0.237187  True         True
           classe   0.80776  0.148424  True         True

Continua a bater a régua: SIM
